In [1]:
import os
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import requests

TMDB_API_KEY = "cc0093b5ad09a876190e097a1c2c8e65"
base_url = "https://api.themoviedb.org/3"

OUTPUT_DIR = ".."

# Make core.game_logic importable (same trick streamlit/app.py uses: put the
# streamlit/ directory itself on sys.path so `core` resolves as a top-level
# package). game_logic.py has no streamlit import, so this works standalone.
sys.path.append("../../streamlit")
from core.game_logic import build_and_save

In [3]:
"""Shared GET helper with retry + exponential backoff.

Retries on network errors and on HTTP 429 (rate limited), honoring TMDB's
Retry-After header when present. All API calls in this notebook go through
this so a burst of 429s during the threaded movie/detail fetches degrades
into a slowdown instead of silently dropped/failed requests.
"""

def request_with_retry(url, params, max_retries=5, backoff_base=1.0, timeout=10):
    for attempt in range(max_retries):
        try:
            response = requests.get(url, params=params, timeout=timeout)
        except requests.exceptions.RequestException:
            if attempt == max_retries - 1:
                raise
            time.sleep(backoff_base * (2 ** attempt))
            continue

        if response.status_code == 429:
            retry_after = response.headers.get('Retry-After')
            wait = float(retry_after) if retry_after else backoff_base * (2 ** attempt)
            print(f"Rate limited by TMDB, waiting {wait:.1f}s (attempt {attempt + 1}/{max_retries})")
            time.sleep(wait)
            continue

        response.raise_for_status()
        return response

    raise requests.exceptions.RequestException(f"Exceeded {max_retries} retries for {url}")

In [4]:
"""API call to retrieve the top N actors directly from TMDB's /person/popular

endpoint. Results are sorted by TMDB's popularity score; only people whose
known_for_department is "Acting" and who have at least one English-language
credit are kept, matching the same filter used for the existing dataset.
"""

def get_top_actors(limit=5000):
    actors = []
    page = 1

    while len(actors) < limit:
        try:
            response = request_with_retry(f"{base_url}/person/popular", params={
                'api_key': TMDB_API_KEY,
                'page': page
            })
            response = response.json()
        except requests.exceptions.RequestException as e:
            print(f"Error: {e}")
            break

        results = response.get('results', [])
        if not results:
            break

        for person in results:
            is_english = any(item.get('original_language') == "en" for item in person.get('known_for', []))
            is_actor = person.get('known_for_department') == "Acting"

            if is_english and is_actor:
                actors.append({
                    'name': person['name'],
                    'tmdb_id': person['id']
                })

            if len(actors) >= limit:
                break

        print(f"Collected: {len(actors)} actors (Page {page})", end='\r')
        page += 1
        time.sleep(0.1)

    return pd.DataFrame(actors)


actor_list = get_top_actors(limit=5000)
actor_list

,name,tmdb_id
0,Julia Doyle,2676077
1,Tom Holland,1136406
2,Matt Damon,1892
3,Alessandra Ambrosio,207606
4,Jason Statham,976
...,...,...
4995,Péter Jankovics,1675710
4996,Jessica Szohr,130782
4997,Li Keyi,5698791
4998,George Cheung,16580


In [5]:
"""Single API call to retrieve the list of movies an actor is in."""

def get_movies(person_id):
    url = f"{base_url}/person/{person_id}/movie_credits"

    try:
        response = request_with_retry(url, params={
            'api_key': TMDB_API_KEY
        })
        response = response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error: {e}")
        return None

    cast_list = pd.DataFrame(response['cast'])
    columns = ['id', 'original_language', 'original_title']
    cast_list = cast_list[columns].rename(columns={'id': 'movie_id'})

    return cast_list

In [6]:
"""Executes get_movies() for the full list of actors.

max_workers is kept modest (rather than firing hundreds of requests at once)
so the pool of concurrent requests stays reasonable; request_with_retry
handles any 429s that still slip through.
"""

def get_all_movies(actor_ids, max_workers=8):
    full_list = []
    total = len(actor_ids)
    completed = 0

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(get_movies, actor_id): actor_id for actor_id in actor_ids}

        for future in as_completed(futures):
            actor_id = futures[future]
            completed += 1

            try:
                result = future.result()
                if result is not None and not result.empty:
                    result['actor_id'] = actor_id
                    full_list.append(result)
                    print(f"[{completed}/{total}] Actor {actor_id} - {len(result)} movies found")
                else:
                    print(f"[{completed}/{total}] Actor {actor_id} - no results")

            except Exception as e:
                print(f"Actor ID: {actor_id} encountered an error: {e}")

    full_df = pd.concat(full_list, ignore_index=True)
    return full_df


expanded_movie_list = get_all_movies(actor_list['tmdb_id'], max_workers=8)

[1/5000] Actor 2638587 - 11 movies found
[2/5000] Actor 976 - 69 movies found
[3/5000] Actor 2676077 - 2 movies found
[4/5000] Actor 1892 - 128 movies found
[5/5000] Actor 207606 - 10 movies found
[6/5000] Actor 947296 - 22 movies found
[7/5000] Actor 31 - 181 movies found
[8/5000] Actor 1136406 - 40 movies found
[9/5000] Actor 2894471 - 3 movies found
[10/5000] Actor 1397778 - 42 movies found
[11/5000] Actor 1245 - 100 movies found
[12/5000] Actor 5081 - 61 movies found
[13/5000] Actor 1493434 - 7 movies found
[14/5000] Actor 505710 - 36 movies found
[15/5000] Actor 1813 - 84 movies found
[16/5000] Actor 64439 - 62 movies found
[17/5000] Actor 11288 - 54 movies found
[18/5000] Actor 19498 - 52 movies found
[19/5000] Actor 1721740 - 16 movies found
[20/5000] Actor 115440 - 55 movies found
[21/5000] Actor 287 - 112 movies found
[22/5000] Actor 126932 - 55 movies found
[23/5000] Actor 18897 - 204 movies found
[24/5000] Actor 64295 - 36 movies found
[25/5000] Actor 500 - 94 movies found
[

In [7]:
movie_list = expanded_movie_list.groupby(['movie_id', 'original_language', 'original_title'])['actor_id']
movie_list = movie_list.apply(lambda x: ",".join(x.astype(str))).reset_index()
movie_list = movie_list.rename(columns={"actor_id": "actors"})
movie_list = movie_list[movie_list['actors'].str.contains(",")]
movie_list = movie_list[movie_list['original_language'] == "en"]
movie_list

,movie_id,original_language,original_title,actors
0,5,en,Four Rooms,"62,3131,3141,3129,3130,3136,3124,3128,3125,313..."
1,6,en,Judgment Night,"5724,12799,9777,10822,2880,11803"
2,11,en,Star Wars,"3,2,4,12248,15152,5,69249"
3,12,en,Finding Nemo,"5293,8783,19,17401,20,118,57675,18,7907,13,613..."
4,13,en,Forrest Gump,"31,6856,35,9640,32,34,33,36221,21457"
...,...,...,...,...
104007,1738103,en,Julie and the Phantoms,"1932318,121868"
104013,1738359,en,Labyrinth,"2478,327,21594,81295"
104015,1738399,en,Star Wars: Maul - Shadow Lord,"52583,352,1217648,98103"
104017,1738470,en,The Winds of Winter,"1223786,1001657"


In [8]:
"""Single API call to get the remaining movie details needed."""

def get_details(movie_id):
    url = f"{base_url}/movie/{movie_id}"

    try:
        response = request_with_retry(url, params={
            'api_key': TMDB_API_KEY
        })
        response = response.json()

        return {
            "id": movie_id,
            "year": response.get("release_date", "")[:4],
            "revenue": response.get("revenue")
        }

    except requests.exceptions.RequestException as e:
        print(f"Error: {e}")
        return {"id": None, "year": None, "revenue": None}

In [9]:
"""max_workers is kept modest here too, for the same reason as get_all_movies."""

def get_all_details(movie_ids, max_workers=10):
    full_data = []
    total = len(movie_ids)
    completed = 0

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(get_details, movie_id): movie_id for movie_id in movie_ids}

        for future in as_completed(futures):
            completed += 1

            try:
                result = future.result()
                full_data.append(result)
            except Exception as e:
                print(f"Error: {e}")

            print(f"[{completed}/{total}]", end='\r')

    df = pd.DataFrame(full_data, columns=["id", "year", "revenue"])
    df = df.sort_values("id").reset_index(drop=True)

    return df


final_movie_details = get_all_details(movie_list["movie_id"], max_workers=10)

In [10]:
inflation_data = pd.read_excel("../BLS Inflation History.xlsx", header=11)
inflation_data['Annual'] = inflation_data['Annual'].fillna(inflation_data['Jan'])
inflation_data = inflation_data[['Year', 'Annual']]
cpi = pd.Series(inflation_data.set_index('Year')['Annual'])

def adjust_for_inflation(value, from_year, to_year=2026, cpi=cpi):
    if from_year == to_year:
        return value

    years = range(from_year + 1, to_year + 1)
    multiplier = 1
    for year in years:
        rate = cpi[year] / 100
        multiplier = multiplier * (1 + rate)

    return round(value * multiplier, 2)

c:\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [11]:
final_movie_list = movie_list.merge(final_movie_details, left_on='movie_id', right_on='id', how="left")
final_movie_list = final_movie_list.drop(columns=["original_language", "id"])
final_movie_list = final_movie_list[final_movie_list["year"] != ""]
final_movie_list = final_movie_list[final_movie_list["year"].isna() == False]
final_movie_list['year'] = final_movie_list['year'].astype(int)
final_movie_list = final_movie_list[final_movie_list['year'] >= 1958]
final_movie_list["adjusted_revenue"] = final_movie_list.apply(lambda row: adjust_for_inflation(row['revenue'], row['year']), axis=1)
final_movie_list

,movie_id,original_title,actors,year,revenue,adjusted_revenue
0,5,Four Rooms,"62,3131,3141,3129,3130,3136,3124,3128,3125,313...",1995,4257354,8.876780e+06
1,6,Judgment Night,"5724,12799,9777,10822,2880,11803",1993,12136938,2.679509e+07
2,11,Star Wars,"3,2,4,12248,15152,5,69249",1977,775398007,4.270401e+09
3,12,Finding Nemo,"5293,8783,19,17401,20,118,57675,18,7907,13,613...",2003,940335536,1.636219e+09
4,13,Forrest Gump,"31,6856,35,9640,32,34,33,36221,21457",1994,677387716,1.454756e+09
...,...,...,...,...,...,...
43506,1737513,Tiada Pengganti MV - Behind the Scenes,"10980,42160",2026,0,0.000000e+00
43508,1737768,Robot Chicken Adult Swim Special,"51798,80757",2026,0,0.000000e+00
43511,1737989,Bad Bunny's Super Bowl LX Halftime Show,"128057,2550161",2026,0,0.000000e+00
43514,1738085,TERM,"2,14329",2026,0,0.000000e+00


In [12]:
final_movie_list.to_parquet(f"{OUTPUT_DIR}/final_movies.parquet")
actor_list.to_parquet(f"{OUTPUT_DIR}/actor_list.parquet")

In [ ]:
"""Build a difficulty-tiered game_data.pkl for each actor-pool size, via
core.game_logic.build_and_save(). Rereads actor_list.parquet from disk
(rather than reusing the in-memory actor_list variable from the cells above)
so this cell is self-contained and can be rerun on its own - after a kernel
restart, or to regenerate tiers - without redoing the API pulls above.

actor_list is already ordered by TMDB popularity (get_top_actors() appends
pages of /person/popular in the order returned), so actor_list.head(n) is
exactly the top n most popular actors.

build_and_save() prunes movies/actors that fall outside the given actor pool
on its own, so reusing the same final_movies.parquet for every tier and only
varying the actor list is enough to produce a correctly-connected sub-graph
per tier.
"""

actor_list = pd.read_parquet(f"{OUTPUT_DIR}/actor_list.parquet")

DIFFICULTY_TIERS = {
    'easy': 500,
    'medium': 1500,
    'hard': 5000,
}

for difficulty, actor_count in DIFFICULTY_TIERS.items():
    tier_actors_path = f"{OUTPUT_DIR}/_tmp_actor_list_{difficulty}.parquet"
    actor_list.head(actor_count).to_parquet(tier_actors_path)

    build_and_save(
        f"{OUTPUT_DIR}/final_movies.parquet",
        tier_actors_path,
        output_path=f"{OUTPUT_DIR}/game_data_{difficulty}.pkl",
    )

    os.remove(tier_actors_path)